# Feature Store Streaming Demo - Monitoring

Observe the live system:
1. Stream feature view freshness
2. Event volume and trending items
3. Online store current state
4. Inference results review

In [ ]:
!pip install --upgrade "snowflake-ml-python>=1.41" snowflake-connector-python requests

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.ml.feature_store import FeatureStore, CreationMode, online_service

session = get_active_session()
session.use_schema("ML_DEMOS.ONLINE_W_STREAMING")
session.use_warehouse("ML_DEMO_WH")

fs = FeatureStore(
    session=session, database="ML_DEMOS", name="ONLINE_W_STREAMING",
    default_warehouse="ML_DEMO_WH", creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)
print("Connected.")

In [ ]:
# Online service status and endpoints
status = fs.get_online_service_status()
print(f"Status: {status.status}")
print(f"Ingest: {online_service.endpoint_url(status, 'ingest')}")
print(f"Query:  {online_service.endpoint_url(status, 'query')}")

In [ ]:
# Event volume and freshness
session.sql('''
    SELECT
        COUNT(*) AS total_events,
        MAX(event_ts) AS latest_event,
        DATEDIFF('second', MAX(event_ts), CURRENT_TIMESTAMP()) AS seconds_since_latest
    FROM RAW_EVENTS
''').show()

In [ ]:
# Event type distribution (last hour)
session.sql('''
    SELECT event_type, COUNT(*) AS cnt,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM RAW_EVENTS
    WHERE event_ts > DATEADD('hour', -1, CURRENT_TIMESTAMP())
    GROUP BY 1 ORDER BY 2 DESC
''').show()

In [ ]:
# Trending items (most viewed in last 5 minutes)
session.sql('''
    SELECT item_id, COUNT(*) AS recent_views
    FROM RAW_EVENTS
    WHERE event_type = 'view'
      AND event_ts > DATEADD('minute', -5, CURRENT_TIMESTAMP())
    GROUP BY 1 ORDER BY 2 DESC LIMIT 10
''').show()

In [ ]:
# Read stream feature view state for sample users
import os
from snowflake.ml.feature_store.feature_view import StoreType

os.environ["SNOWFLAKE_PAT"] = session.connection.rest.token

stream_fv = fs.get_feature_view("USER_EVENTS_STREAM_FV", "V1")

# Pick some active user-item pairs
active_pairs = session.sql('''
    SELECT DISTINCT user_id, item_id FROM RAW_EVENTS
    WHERE event_ts > DATEADD('minute', -10, CURRENT_TIMESTAMP())
      AND item_id IS NOT NULL
    LIMIT 5
''').collect()

if active_pairs:
    user_item_keys = [[row[0], row[1]] for row in active_pairs]
    print(f"Querying stream FV for user-item pairs: {user_item_keys}")
    online_df = fs.read_feature_view(stream_fv, keys=user_item_keys, store_type=StoreType.ONLINE)
    print(online_df)
else:
    print("No recent active user-item pairs. Run simulate_events.py first.")

In [ ]:
# Inference results (if run_inference.py has been writing results)
try:
    session.sql('''
        SELECT user_id, recommended_items, scores, inference_ts
        FROM INFERENCE_RESULTS
        ORDER BY inference_ts DESC
        LIMIT 10
    ''').show()
except Exception as e:
    print(f"No inference results table yet: {e}")
    print("Run run_inference.py to generate recommendations.")

In [ ]:
# Active sessions in the last 5 minutes
session.sql('''
    SELECT COUNT(DISTINCT session_id) AS active_sessions,
           COUNT(DISTINCT user_id) AS active_users
    FROM RAW_EVENTS
    WHERE event_ts > DATEADD('minute', -5, CURRENT_TIMESTAMP())
''').show()